# DermaSalient v2 — Research Notebook
**Advanced Medical Saliency Detection on ISIC 2020 Melanoma Dataset**

This notebook walks through the complete pipeline from data exploration to
statistical benchmarking.  Each section explains the research motivation
before the code so that the choices are transparent and reproducible.

---
**Dataset:** ISIC 2020 Melanoma Classification + ISIC 2018 Segmentation Masks  
**Backbone:** EfficientNet-B4 (timm)  
**Saliency methods:** 9 (6 CAM + U²-Net + SAM + Ensemble)  
**Evaluation:** 7 SOD metrics against real clinical masks

## Section 0 — Setup, GPU Check, Config

We seed every random number generator for reproducibility and print
the device we will run on.  All hyperparameters live in `src/utils/config.py`
so that changing them propagates everywhere without searching the codebase.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
%matplotlib inline
matplotlib.rcParams['figure.dpi'] = 120

from src.utils.config import seed_everything, DEVICE, TRAIN_CSV, IMG_DIR, MASK_DIR
seed_everything()

print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU             : {torch.cuda.get_device_name(0)}')
    print(f'VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'Device          : {DEVICE}')

## Section 1 — Data Exploration

Understanding the data before modelling is mandatory in medical imaging.
Key questions:
- How severe is the class imbalance?
- What does a typical benign vs. malignant lesion look like?
- How many images have ground-truth masks?

The ISIC 2020 dataset has ~33k images with 1.8 % malignant prevalence
(we subsample to N_BENIGN=2000 / N_MALIGNANT=584 for tractable training).

In [ ]:
df = pd.read_csv(TRAIN_CSV)

print(f'Total images    : {len(df):,}')
print(f'Malignant       : {df["target"].sum():,}  ({df["target"].mean()*100:.1f}%)')
print(f'Benign          : {(1-df["target"]).sum():,}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Class distribution
counts = df['target'].value_counts()
axes[0].bar(['Benign', 'Malignant'], counts.values, color=['steelblue', 'tomato'])
axes[0].set_title('Class Distribution (full ISIC 2020)')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 100, f'{v:,}', ha='center', fontsize=10)

# Age distribution by class
for label, colour in [(0, 'steelblue'), (1, 'tomato')]:
    sub = df[df['target'] == label]['age_approx'].dropna()
    name = 'Benign' if label == 0 else 'Malignant'
    axes[1].hist(sub, bins=20, alpha=0.6, label=name, color=colour)
axes[1].set_xlabel('Age'); axes[1].set_ylabel('Count')
axes[1].set_title('Age Distribution by Class')
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
import cv2
from src.data.masks import count_masks, load_mask

mask_stats = count_masks()
print(f'Ground-truth masks available: {mask_stats["total"]}  (in {mask_stats["dir"]})')

# Sample grid: 4 benign + 4 malignant
benign    = df[df['target'] == 0]['image_name'].iloc[:4].tolist()
malignant = df[df['target'] == 1]['image_name'].iloc[:4].tolist()

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('Sample Images — Top: Benign | Bottom: Malignant', fontsize=13)

for i, (names, label) in enumerate([(benign, 'Benign'), (malignant, 'Malignant')]):
    for j, name in enumerate(names):
        path = os.path.join(IMG_DIR, f'{name}.jpg')
        img  = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB) if os.path.exists(path) else np.zeros((224,224,3), np.uint8)
        mask = load_mask(name)
        ax   = axes[i, j]
        ax.imshow(img)
        if mask is not None:
            contours, _ = cv2.findContours((mask*255).astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            ax.set_title(f'{name[:12]}\n(mask ✓)', fontsize=7)
        else:
            ax.set_title(f'{name[:12]}', fontsize=7)
        ax.axis('off')
plt.tight_layout()
plt.show()

## Section 2 — Train Classifier

We train EfficientNet-B4 with several research-grade decisions:

1. **WeightedRandomSampler** — ensures balanced mini-batches without oversampling
2. **Pos-weight BCE** — additional weight on the minority class in the loss
3. **Label smoothing** — reduces overconfidence, improves calibration
4. **CosineAnnealingWarmRestarts** — cyclical LR avoids sharp local minima
5. **Early stopping on AUC-ROC** — not accuracy (misleading on imbalanced data)

AUC-ROC is the standard metric for medical classification because it measures
discrimination across ALL thresholds, not just the 0.5 default.

In [ ]:
from src.data.dataset import build_dataloaders
from src.models.classifier import build_classifier, train as train_classifier

train_loader, val_loader, test_loader = build_dataloaders(num_workers=0)
print(f'Train batches: {len(train_loader)}')
print(f'Val   batches: {len(val_loader)}')
print(f'Test  batches: {len(test_loader)}')

# Verify WeightedRandomSampler is working (batch class balance)
batch = next(iter(train_loader))
labels = batch['label'].numpy()
print(f'\nSample batch label distribution: {dict(zip(*np.unique(labels, return_counts=True)))}')

In [ ]:
# ⚠ This cell trains the classifier (~30 epochs).  Skip if weights already exist.
from src.utils.config import CLASSIFIER_CKPT

if os.path.isfile(CLASSIFIER_CKPT):
    print(f'Classifier already trained: {CLASSIFIER_CKPT}')
    print('Delete the checkpoint and re-run to retrain.')
else:
    model   = build_classifier(pretrained=True)
    history = train_classifier(model, train_loader, val_loader)

In [ ]:
from src.utils.config import REPORT_DIR
from src.utils.visualization import plot_training_history

hist_path = os.path.join(REPORT_DIR, 'training_history.csv')
if os.path.isfile(hist_path):
    hist_df = pd.read_csv(hist_path)
    fig = plot_training_history(hist_df)
    plt.show()
    print(f'Best val AUC: {hist_df["va_auc"].max():.4f} (epoch {hist_df["va_auc"].idxmax()+1})')
else:
    print('No training history found.  Train the classifier first.')

## Section 3 — Saliency Map Generation

We generate saliency maps using 9 different methods and visualise them side
by side on 10 test images.  The key insight is that different methods capture
different aspects of the network's decision:

- **GradCAM / GradCAM++** — gradient-weighted feature maps; fast, noisy
- **ScoreCAM** — perturbation-based; smoother, no gradient artefacts
- **EigenCAM** — PCA of feature maps; no backprop needed; interpretable
- **U²-Net** — dedicated SOD model; class-agnostic, best at boundaries
- **SAM** — foundation model for segmentation; GradCAM peak as prompt
- **Ensemble** — weighted fusion; consistently outperforms individual methods

In [ ]:
from src.models.classifier import load_classifier
from src.saliency.cam_methods import run_all_cams
from src.saliency.u2net_infer import u2net_saliency_from_tensor
from src.saliency.sam_infer import sam_saliency
from src.saliency.fusion import ensemble_saliency
from src.saliency.postprocess import full_postprocess_pipeline
from src.data.augmentations import val_transforms
from src.utils.visualization import saliency_gallery
from src.data.masks import load_mask

classifier = load_classifier()
tf         = val_transforms()

# Pick 10 test images
from src.data.dataset import build_splits
_, _, test_df = build_splits()
sample_images = test_df['image_name'].iloc[:10].tolist()

In [ ]:
for image_name in sample_images[:3]:    # show 3 galleries to keep notebook size manageable
    path    = os.path.join(IMG_DIR, f'{image_name}.jpg')
    img_bgr = cv2.imread(path)
    if img_bgr is None:
        print(f'Image not found: {path}')
        continue
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    h, w    = img_rgb.shape[:2]
    tensor  = tf(image=img_rgb)['image'].to(DEVICE)

    cams    = run_all_cams(classifier, tensor, target_size=(h, w))
    try:
        u2_sal = u2net_saliency_from_tensor(tensor)
    except Exception:
        u2_sal = cams['gradcam']
    ens_sal = ensemble_saliency({**cams, 'u2net': u2_sal})

    all_sals = {**cams, 'u2net': u2_sal, 'ensemble': ens_sal}
    gt_mask  = load_mask(image_name)

    _, bin_maps = zip(*[
        full_postprocess_pipeline(img_rgb, sal)
        for sal in all_sals.values()
    ])
    binary_maps = dict(zip(all_sals.keys(), bin_maps))

    fig = saliency_gallery(img_rgb, all_sals, binary_maps, gt_mask)
    fig.suptitle(f'{image_name}  (GT mask: {"yes" if gt_mask is not None else "no"})', fontsize=11)
    plt.show()
    plt.close()

## Section 4 — Benchmarking + Leaderboard

We evaluate every method against real clinical segmentation masks from
ISIC 2018.  Using Otsu threshold on the raw image as pseudo-ground-truth
(as done in v1) is not a valid evaluation — it just measures how well
each method agrees with Otsu, not how clinically accurate it is.

The 7 metrics cover different failure modes:
- **MAE** — average per-pixel error (penalises diffuse predictions)
- **F-measure** — precision/recall trade-off at adaptive threshold
- **Weighted F** — addresses interpolation flaw in standard F-measure
- **S-measure** — structural similarity (object + region aware)
- **E-measure** — pixel + image level alignment
- **Dice / IoU** — standard medical segmentation overlap metrics

In [ ]:
from src.utils.config import BENCH_DIR

results_csv = os.path.join(BENCH_DIR, 'full_results.csv')
if not os.path.isfile(results_csv):
    print('Run benchmark.py first:  python benchmark.py')
else:
    df  = pd.read_csv(results_csv)
    METRICS = ['mae', 'f_measure', 'weighted_f', 's_measure', 'e_measure', 'dice', 'iou']

    lb = (
        df.groupby('model')[METRICS]
        .mean()
        .sort_values('f_measure', ascending=False)
        .round(4)
    )
    print('LEADERBOARD (ranked by F-measure):')
    display(lb)

In [ ]:
if os.path.isfile(results_csv):
    from src.utils.visualization import benchmark_bar_chart
    for metric in ['f_measure', 'dice', 'iou', 'mae']:
        fig = benchmark_bar_chart(df, metric)
        fig.update_layout(height=350)
        fig.show()

## Section 5 — Statistical Significance Analysis

A benchmark leaderboard is meaningless without significance testing.
The Wilcoxon signed-rank test is the standard non-parametric alternative
to the paired t-test when the data may not be normally distributed
(which per-image IoU scores often are not).

We report both p-values and effect sizes (rank-biserial correlation),
and compute 95% bootstrap confidence intervals for all metrics.

In [ ]:
sig_csv = os.path.join(BENCH_DIR, 'significance_table.csv')
if os.path.isfile(sig_csv):
    sig_df = pd.read_csv(sig_csv)
    sig_count = sig_df['significant'].sum()
    print(f'Significant pairs (p < 0.05): {sig_count}/{len(sig_df)}')
    display(sig_df.style.background_gradient(
        cmap='RdYlGn_r', subset=['p_value'], vmin=0, vmax=0.1
    ))
else:
    print('Run benchmark.py first.')

In [ ]:
if os.path.isfile(sig_csv):
    from src.utils.visualization import significance_heatmap
    fig = significance_heatmap(sig_df)
    plt.show()
    plt.close()

In [ ]:
ci_csv = os.path.join(BENCH_DIR, 'confidence_intervals.csv')
if os.path.isfile(ci_csv):
    ci_df = pd.read_csv(ci_csv)
    f_ci  = ci_df[ci_df['metric'] == 'f_measure'].copy()
    f_ci['ci_str'] = f_ci.apply(
        lambda r: f"{r['mean']:.4f} [{r['lower']:.4f}, {r['upper']:.4f}]", axis=1
    )
    print('F-measure 95% Bootstrap Confidence Intervals:')
    display(f_ci[['model', 'ci_str']].rename(columns={'ci_str': 'mean [95% CI}'}))
else:
    print('Run benchmark.py first.')

## Section 6 — Ensemble Fusion Analysis

The ensemble method combines all 9 individual saliency maps via weighted
averaging.  We show two things:

1. Ensemble F-measure vs. best individual method — does it consistently win?
2. Ablation: equal weights vs. performance-weighted fusion (learned weights)

Committee-of-experts ensembles typically outperform individual methods because
they smooth over method-specific failure modes (e.g., GradCAM is noisy on
texturally complex lesions; U²-Net struggles on low-contrast boundaries).

In [ ]:
if os.path.isfile(results_csv):
    df = pd.read_csv(results_csv)

    # Per-image: does ensemble beat the best CAM method?
    pivot = df.pivot_table(index='image_name', columns='model', values='f_measure')
    cam_methods = ['gradcam', 'gradcam_pp', 'scorecam', 'layercam', 'eigencam', 'xgradcam']
    avail_cams  = [m for m in cam_methods if m in pivot.columns]

    if 'ensemble' in pivot.columns and avail_cams:
        best_cam = pivot[avail_cams].max(axis=1)
        ensemble_wins = (pivot['ensemble'] > best_cam).sum()
        n = len(pivot.dropna(subset=['ensemble']))
        print(f'Ensemble beats best individual CAM in {ensemble_wins}/{n} images '
              f'({ensemble_wins/n*100:.1f}%)')

        fig, ax = plt.subplots(figsize=(8, 4))
        ax.scatter(best_cam, pivot['ensemble'], alpha=0.4, s=10)
        lo = min(best_cam.min(), pivot['ensemble'].min()) * 0.95
        hi = max(best_cam.max(), pivot['ensemble'].max()) * 1.05
        ax.plot([lo, hi], [lo, hi], 'r--', label='y=x (parity)')
        ax.set_xlabel('Best individual CAM F-measure')
        ax.set_ylabel('Ensemble F-measure')
        ax.set_title('Ensemble vs. Best Individual CAM (per image)')
        ax.legend()
        plt.tight_layout()
        plt.show()
    else:
        print('Ensemble or CAM results not found in results CSV.')
else:
    print('Run benchmark.py first.')

## Section 7 — Error Analysis

Examining failure cases is as important as reporting top-line metrics.
Common failure modes in dermoscopy saliency:

1. **Ruler/artefact highlighting** — GradCAM sometimes focuses on calibration
   rulers rather than lesions
2. **Low-contrast margins** — U²-Net struggles when lesion merges with
   normal skin
3. **Multi-lesion images** — SAM point-prompted can select the wrong lesion
   if the GradCAM prompt is off-target
4. **Dark background** — EigenCAM is sensitive to frame colour

We identify the worst-performing images by F-measure and visualise them.

In [ ]:
if os.path.isfile(results_csv):
    df = pd.read_csv(results_csv)

    # Find images where ensemble F-measure < 0.3
    ens = df[df['model'] == 'ensemble'].sort_values('f_measure')
    worst = ens.head(5)['image_name'].tolist()

    print('Worst 5 images (ensemble F-measure):')
    display(ens.head(5)[['image_name', 'f_measure', 'dice', 'iou']])

    fig, axes = plt.subplots(2, min(5, len(worst)), figsize=(15, 6))
    for j, name in enumerate(worst[:5]):
        path    = os.path.join(IMG_DIR, f'{name}.jpg')
        img     = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB) if os.path.exists(path) else np.zeros((256,256,3), np.uint8)
        gt_mask = load_mask(name)

        axes[0, j].imshow(img)
        axes[0, j].set_title(f'{name[:12]}', fontsize=8)
        axes[0, j].axis('off')

        if gt_mask is not None:
            axes[1, j].imshow(gt_mask, cmap='gray')
            axes[1, j].set_title('GT Mask', fontsize=8)
        else:
            axes[1, j].axis('off')
        axes[1, j].axis('off')

    plt.suptitle('Failure Cases — Low Ensemble F-measure', fontsize=12)
    plt.tight_layout()
    plt.show()
else:
    print('Run benchmark.py first.')

## Section 8 — Launch App

The Gradio application provides an interactive research interface with four tabs:
- **Single Image Analysis** — any method, CRF on/off, TTA on/off
- **Model Comparison** — all 9 methods side-by-side on one image
- **Benchmark Results** — interactive leaderboard and significance tables
- **Batch Processing** — process an entire folder of images

The `share=True` flag creates a public URL for 72-hour demo sharing.

In [ ]:
# Launch the app from inside the notebook
import subprocess, sys
print('Launching DermaSalient v2 app...')
print('Go to: http://localhost:7860')

# Run as a subprocess so it doesn't block the notebook
proc = subprocess.Popen([sys.executable, '../app/app.py'])
print(f'App started (PID {proc.pid}).  Stop with proc.terminate()')